In [ ]:
# Less Salt Less Sugar Monthly Report (launch from 31Aug2026. last for 1 year at least)
#     1. Daily Badge impression by device 
#     2. Daily Brand Page (LMS) Pageview/ Impressions     
#     3. Daily Theme Listing Impressions           --> search --> 少鹽少糖

In [ ]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ2.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [ ]:
current_date = datetime.date.today()
first_day_of_current_month = datetime.date(current_date.year, current_date.month, 1)
last_day_of_previous_month = first_day_of_current_month - datetime.timedelta(days=1)

month = last_day_of_previous_month.month
year = last_day_of_previous_month.year

str_month = str(month)
if len(str_month)==1:
    str_month = '0'+str_month
str_month

In [ ]:
template_file = 'report_for_BD_template.xlsx'
excel_name = template_file.replace('template.xlsx', '%s.xlsx' % datetime.date.today())
copyfile(template_file, excel_name)

In [ ]:
# 1. Daily Badge impression by device     DONE
#  --> monthly_icon_impression_report_result.ipynb cell 8 少鹽少糖食店 Icon Impression v2
# 
# #少鹽少糖食店 Icon Impression v2

sql = f'''

SELECT date(time) as querydate,platform,count(1) as count_ FROM `openrice-production.ORGA.PV_{year}{str_month}*` 
WHERE EventAction = 'impression.poi'
and cast(REGEXP_EXTRACT(lower(EventLabelRaw), r'poiid:(\d+)') as INT64) in  (Select poiid FROM `openrice-production.openrice3.promotionpoi` WHERE PromotionId =11)
group by platform,querydate
    '''

df_big_query = client.query(sql).result().to_dataframe()

pivoted_df = df_big_query.pivot(index='querydate', columns='platform', values='count_')
pivoted_df = pivoted_df.fillna(0)
pivoted_df["Web"]=pivoted_df["desktop"]
pivoted_df["Mobile Web"]=pivoted_df["mobile"]
try:
    pivoted_df["Android"]=pivoted_df["android"]+pivoted_df["hms"]
except:
    pivoted_df["Android"]=pivoted_df["android"]
    
pivoted_df["IOS"]=pivoted_df["ios"]
pivoted_df =pivoted_df[["Web","Mobile Web","Android","IOS"]]

pivoted_df.index.name = None
pivoted_df.reset_index(inplace=True)

with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
#     book = load_workbook(excel_name)
#     writer.book = book
#     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
    pivoted_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=7, header=None, index=False)

In [ ]:
# 2. Daily Brand Page (LMS) Pageview       no date, group by web & app
# 17:37:14|| view.SR1.Promotion| CityID:0;PromotionID:13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C12%2C13%2C13%2C13%2C13%2C13%2C13;;Lang:zh_TW;Ver:7.20.4; sn:hk.LMS2.35336.tab.-990.1


sql = f"""
with theme_list as (
    select platform
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (LOWER(EventAction) = 'view.sr1.promotion'
    AND ((LOWER(EventLabelRaw) LIKE '%lms2%') or (LOWER(EventLabelRaw) LIKE '%promotionid:13%')))
)

select platform, count(1) as count
from theme_list
group by platform
"""

df_big_query_3 = client.query(sql).result().to_dataframe()

web = df_big_query_3.query("platform=='mobile' | platform=='desktop'  ")["count"].sum()
app = df_big_query_3.query("platform=='android' | platform=='ios' | platform=='hms' ")["count"].sum()
temp_df = pd.DataFrame({'date':[f'{year}-{str_month}'],'web':[web],'app':[app]})

temp_df

# with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
# #     book = load_workbook(excel_name)
# #     writer.book = book
# #     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
#     temp_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=42, header=None, index=False)


In [ ]:
df_big_query_3

In [ ]:
# 2. Daily Brand Page (LMS) Pageview   hv date, group by devices

sql = f"""
    select date(time) as querydate, platform, count(1) as count
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (
        LOWER(EventAction) = 'view.sr1.promotion'
        AND (
            (LOWER(EventLabelRaw) LIKE '%lms2%') or (LOWER(EventLabelRaw) LIKE '%promotionid:13%')
            )
        )
    group by platform, querydate
    """


df_big_query_6 = client.query(sql).result().to_dataframe()

pivoted_df = df_big_query_6.pivot(index='querydate', columns='platform', values='count')
pivoted_df = pivoted_df.fillna(0)
pivoted_df["Web"]=pivoted_df["desktop"]
pivoted_df["Mobile Web"]=pivoted_df["mobile"]
try:
    pivoted_df["Android"]=pivoted_df["android"]+pivoted_df["hms"]
except:
    pivoted_df["Android"]=pivoted_df["android"]
    
pivoted_df["IOS"]=pivoted_df["ios"]
pivoted_df =pivoted_df[["Web","Mobile Web","Android","IOS"]]

pivoted_df.index.name = None
pivoted_df.reset_index(inplace=True)

pivoted_df
# with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
# #     book = load_workbook(excel_name)
# #     writer.book = book
# #     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
#     pivoted_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=42, header=None, index=False)

In [ ]:
# 3. Daily Theme Listing Impressions
# 17:37:14|| or.search.layer.search| CityID:0;geo:22.2915161%2C114.2081815;LndID:35336;Page:1;sr:lmsSr1;Lang:zh_TW;Ver:7.20.4; sn:hk.Search.layer

sql = f"""
    select date(time) as querydate, platform, count(1) as count
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (
        LOWER(EventAction) = 'view.sr1.promotion'
        AND (
            (LOWER(EventLabelRaw) LIKE '%lms2%') or (LOWER(EventLabelRaw) LIKE '%promotionid:13%')
            )
        )
    group by platform, querydate
    """


df_big_query_6 = client.query(sql).result().to_dataframe()

pivoted_df = df_big_query_6.pivot(index='querydate', columns='platform', values='count')
pivoted_df = pivoted_df.fillna(0)
pivoted_df["Web"]=pivoted_df["desktop"]
pivoted_df["Mobile Web"]=pivoted_df["mobile"]
try:
    pivoted_df["Android"]=pivoted_df["android"]+pivoted_df["hms"]
except:
    pivoted_df["Android"]=pivoted_df["android"]
    
pivoted_df["IOS"]=pivoted_df["ios"]
pivoted_df =pivoted_df[["Web","Mobile Web","Android","IOS"]]

pivoted_df.index.name = None
pivoted_df.reset_index(inplace=True)

pivoted_df

# with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
# #     book = load_workbook(excel_name)
# #     writer.book = book
# #     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
#     pivoted_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=77, header=None, index=False)